#### Environment Setup

In [1]:
import os
import math
import json
import gc
import random
import warnings
import inspect
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_callback import ProgressCallback

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name       :", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu126
CUDA available : True
GPU name       : NVIDIA GeForce RTX 4080 Laptop GPU


#### Configuration

In [2]:
CSV_PATH = "discharge.csv"
TEXT_COLUMN = "text"
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
OUTPUT_DIR = "./clinical_bert_results"

TEST_SIZE = 0.10
VAL_SIZE = 0.10
SEED = 42
RANDOM_STATE = 42

BLOCK_SIZE = 128
MLM_PROBABILITY = 0.15

NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
LOGGING_STEPS = 50
SAVE_TOTAL_LIMIT = 1

MAX_ROWS = 1000
NUM_PROC = max(1, min(4, os.cpu_count()))
DATALOADER_NUM_WORKERS = min(4, os.cpu_count())

USE_TORCH_COMPILE = False
PIN_MEMORY = torch.cuda.is_available()

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

TOP_K_VALUES = [1, 3, 5]

#### Device and Reproducibility

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Using device:", device)
print("BF16 enabled:", USE_BF16)
print("FP16 enabled:", USE_FP16)

Using device: cuda
BF16 enabled: True
FP16 enabled: False


#### Utility Functions

In [4]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\x00", " ")
    x = x.replace("\r", " ").replace("\n", " ")
    x = " ".join(x.split())
    return x.strip()

def compute_perplexity(loss_value):
    try:
        return float(math.exp(loss_value))
    except OverflowError:
        return float("inf")

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def tokenize_and_chunk(dataset_dict, tokenizer, text_column="text", block_size=128, num_proc=1):
    def tokenize_function(examples):
        return tokenizer(
            examples[text_column],
            add_special_tokens=True,
            truncation=False,
            return_special_tokens_mask=True
        )

    tokenized = dataset_dict.map(
        tokenize_function,
        batched=True,
        remove_columns=[text_column],
        num_proc=num_proc
    )

    def group_texts(examples):
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated_examples["input_ids"])
        total_length = (total_length // block_size) * block_size
        result = {
            k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated_examples.items()
        }
        return result

    lm_dataset = tokenized.map(group_texts, batched=True, num_proc=num_proc)
    return lm_dataset

#### Evaluation Functions

In [5]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.detach()

def compute_basic_eval_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    preds = preds.reshape(-1)
    labels = labels.reshape(-1)

    valid_mask = labels != -100
    preds_valid = preds[valid_mask]
    labels_valid = labels[valid_mask]

    accuracy = (preds_valid == labels_valid).mean()
    return {"accuracy": float(accuracy)}

@torch.no_grad()
def evaluate_mlm_full_metrics(trainer, eval_dataset, split_name="Validation", topk_values=(1, 3, 5)):
    model = trainer.model
    model.eval()

    eval_dataloader = trainer.get_eval_dataloader(eval_dataset)

    total_loss = 0.0
    total_batches = 0
    total_masked = 0
    total_correct = 0
    topk_correct = {k: 0 for k in topk_values}

    all_preds = []
    all_labels = []

    autocast_dtype = torch.bfloat16 if USE_BF16 else torch.float16

    for step, batch in enumerate(eval_dataloader):
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=autocast_dtype, enabled=torch.cuda.is_available()):
            outputs = model(**batch)

        loss = outputs.loss
        logits = outputs.logits
        labels = batch["labels"]

        total_loss += loss.item()
        total_batches += 1

        mask = labels != -100
        num_masked = mask.sum().item()

        if num_masked == 0:
            continue

        masked_logits = logits[mask]
        masked_labels = labels[mask]

        preds = torch.argmax(masked_logits, dim=-1)

        total_correct += (preds == masked_labels).sum().item()
        total_masked += num_masked

        all_preds.append(preds.detach().cpu())
        all_labels.append(masked_labels.detach().cpu())

        max_k = max(topk_values)
        topk_preds = torch.topk(masked_logits, k=max_k, dim=-1).indices

        for k in topk_values:
            correct_k = (topk_preds[:, :k] == masked_labels.unsqueeze(1)).any(dim=1).sum().item()
            topk_correct[k] += correct_k

    avg_loss = total_loss / max(total_batches, 1)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    accuracy = total_correct / total_masked
    micro_f1 = f1_score(all_labels, all_preds, average="micro")
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    metrics = {
        f"{split_name} Loss": round(avg_loss, 4),
        f"{split_name} Accuracy": round(float(accuracy), 4),
        f"{split_name} Micro F1": round(float(micro_f1), 4),
        f"{split_name} Macro F1": round(float(macro_f1), 4),
    }

    for k in topk_values:
        metrics[f"{split_name} Precision@{k}"] = round(float(topk_correct[k] / total_masked), 4)

    return metrics

#### Data Loading

In [6]:
df = pd.read_csv(CSV_PATH)

required_columns = [
    "note_id",
    "subject_id",
    "hadm_id",
    "note_type",
    "note_seq",
    "charttime",
    "storetime",
    TEXT_COLUMN
]

missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df[TEXT_COLUMN] = df[TEXT_COLUMN].apply(clean_text)
df = df[df[TEXT_COLUMN].str.len() > 0].reset_index(drop=True)

if MAX_ROWS is not None:
    df = df.iloc[:MAX_ROWS].copy()

print("Dataset shape:", df.shape)
display(df[[TEXT_COLUMN]].head())

Dataset shape: (1000, 8)


,text
0,Name: ___ Unit No: ___ Admission Date: ___ Dis...
1,Name: ___ Unit No: ___ Admission Date: ___ Dis...
2,Name: ___ Unit No: ___ Admission Date: ___ Dis...
3,Name: ___ Unit No: ___ Admission Date: ___ Dis...
4,Name: ___ Unit No: ___ Admission Date: ___ Dis...


#### Data Splitting

In [7]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

val_relative = VAL_SIZE / (1 - TEST_SIZE)

train_df, val_df = train_test_split(
    train_df,
    test_size=val_relative,
    random_state=RANDOM_STATE,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size      :", len(train_df))
print("Validation size :", len(val_df))
print("Test size       :", len(test_df))

Train size      : 800
Validation size : 100
Test size       : 100


#### Dataset Conversion

In [8]:
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df[[TEXT_COLUMN]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[[TEXT_COLUMN]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[[TEXT_COLUMN]], preserve_index=False),
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 100
    })
    test: Dataset({
        features: ['text'],
        num_rows: 100
    })
})

#### Model and Tokenizer

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

mlm_model = mlm_model.to(device)

if USE_TORCH_COMPILE and hasattr(torch, "compile"):
    try:
        import triton
        mlm_model = torch.compile(mlm_model)
        print("torch.compile enabled")
    except Exception as e:
        print("torch.compile disabled:", e)

print("Model loaded on:", device)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda


#### Tokenization

In [10]:
lm_dataset = tokenize_and_chunk(
    dataset_dict=dataset_dict,
    tokenizer=tokenizer,
    text_column=TEXT_COLUMN,
    block_size=BLOCK_SIZE,
    num_proc=NUM_PROC
)

print("Train blocks      :", len(lm_dataset["train"]))
print("Validation blocks :", len(lm_dataset["validation"]))
print("Test blocks       :", len(lm_dataset["test"]))

Setting TOKENIZERS_PARALLELISM=false for forked processes.


Map (num_proc=4):   0%|          | 0/800 [00:00<?, ? examples/s]

Setting TOKENIZERS_PARALLELISM=false for forked processes.


Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Setting TOKENIZERS_PARALLELISM=false for forked processes.


...

Setting TOKENIZERS_PARALLELISM=false for forked processes.


Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Setting TOKENIZERS_PARALLELISM=false for forked processes.


Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Train blocks      : 19801
Validation blocks : 2611
Test blocks       : 2320


#### Data Collator

In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY
)

#### Training Arguments

In [12]:
train_dir = os.path.join(OUTPUT_DIR, "mlm_training")
os.makedirs(train_dir, exist_ok=True)

training_kwargs = dict(
    output_dir=train_dir,
    do_train=True,
    do_eval=False,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    fp16=USE_FP16,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=PIN_MEMORY,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    disable_tqdm=False
)

sig = inspect.signature(TrainingArguments.__init__).parameters

if "save_strategy" in sig:
    training_kwargs["save_strategy"] = "epoch"

if "bf16" in sig:
    training_kwargs["bf16"] = USE_BF16

if "eval_accumulation_steps" in sig:
    training_kwargs["eval_accumulation_steps"] = 4

training_args = TrainingArguments(**training_kwargs)
print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=4,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=True,
enable_jit_checkpoint=False,
eval_accumulation_steps=4,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,
fp16

#### Trainer Initialization

In [13]:
optimizer_cls = torch.optim.AdamW
optimizer_kwargs = {
    "lr": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY
}

if torch.cuda.is_available():
    try:
        optimizer_kwargs["fused"] = True
        print("Using fused AdamW")
    except Exception:
        print("Fused AdamW not available")

optimizer = optimizer_cls(mlm_model.parameters(), **optimizer_kwargs)

trainer = Trainer(
    model=mlm_model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["validation"],
    data_collator=data_collator,
    optimizers=(optimizer, None),
    compute_metrics=compute_basic_eval_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

try:
    trainer.add_callback(ProgressCallback)
except Exception:
    pass

print("Trainer initialized")

Using fused AdamW
Trainer initialized


#### Model Training

In [14]:
train_output = trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "best_model"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))

print("Training completed")
print(train_output.metrics)

  0%|                                                                                             | 0/1238 [00…

{'loss': '4.691', 'grad_norm': '9.205', 'learning_rate': '1.581e-05', 'epoch': '0.04039'}
{'loss': '3.688', 'grad_norm': '8.701', 'learning_rate': '1.937e-05', 'epoch': '0.08078'}
{'loss': '3.221', 'grad_norm': '7.206', 'learning_rate': '1.852e-05', 'epoch': '0.1212'}
{'loss': '2.982', 'grad_norm': '7.898', 'learning_rate': '1.767e-05', 'epoch': '0.1616'}
{'loss': '2.94', 'grad_norm': '7.624', 'learning_rate': '1.682e-05', 'epoch': '0.2019'}
{'loss': '2.822', 'grad_norm': '6.499', 'learning_rate': '1.597e-05', 'epoch': '0.2423'}
{'loss': '2.696', 'grad_norm': '7.358', 'learning_rate': '1.512e-05', 'epoch': '0.2827'}
{'loss': '2.711', 'grad_norm': '7.026', 'learning_rate': '1.427e-05', 'epoch': '0.3231'}
{'loss': '2.634', 'grad_norm': '7.165', 'learning_rate': '1.342e-05', 'epoch': '0.3635'}
{'loss': '2.618', 'grad_norm': '6.315', 'learning_rate': '1.257e-05', 'epoch': '0.4039'}
{'loss': '2.582', 'grad_norm': '7.548', 'learning_rate': '1.172e-05', 'epoch': '0.4443'}
{'loss': '2.566', 'g

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '269.5', 'train_samples_per_second': '73.48', 'train_steps_per_second': '4.594', 'train_loss': '2.71', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed
{'train_runtime': 269.4758, 'train_samples_per_second': 73.48, 'train_steps_per_second': 4.594, 'total_flos': 1302906418424832.0, 'train_loss': 2.7097050446493367, 'epoch': 1.0}


#### Evaluation 

In [15]:
small_train_size = min(100, len(lm_dataset["train"]))
small_train_dataset = lm_dataset["train"].select(range(small_train_size))

clear_cuda()
train_metrics_full = evaluate_mlm_full_metrics(
    trainer,
    eval_dataset=small_train_dataset,
    split_name="Training",
    topk_values=TOP_K_VALUES
)

clear_cuda()
val_metrics_full = evaluate_mlm_full_metrics(
    trainer,
    eval_dataset=lm_dataset["validation"],
    split_name="Validation",
    topk_values=TOP_K_VALUES
)

clear_cuda()
test_metrics_full = evaluate_mlm_full_metrics(
    trainer,
    eval_dataset=lm_dataset["test"],
    split_name="Testing",
    topk_values=TOP_K_VALUES
)

clear_cuda()

print("Training metrics  :", train_metrics_full)
print("Validation metrics:", val_metrics_full)
print("Testing metrics   :", test_metrics_full)

Training metrics  : {'Training Loss': 1.3057, 'Training Accuracy': 0.7374, 'Training Micro F1': 0.7374, 'Training Macro F1': 0.5291, 'Training Precision@1': 0.738, 'Training Precision@3': 0.8352, 'Training Precision@5': 0.8592}
Validation metrics: {'Validation Loss': 1.0913, 'Validation Accuracy': 0.7656, 'Validation Micro F1': 0.7656, 'Validation Macro F1': 0.517, 'Validation Precision@1': 0.7657, 'Validation Precision@3': 0.8558, 'Validation Precision@5': 0.8849}
Testing metrics   : {'Testing Loss': 1.071, 'Testing Accuracy': 0.7647, 'Testing Micro F1': 0.7647, 'Testing Macro F1': 0.5129, 'Testing Precision@1': 0.7645, 'Testing Precision@3': 0.8576, 'Testing Precision@5': 0.8868}


#### Final Metrics

In [16]:
final_results = {
    "Training Loss": train_metrics_full["Training Loss"],
    "Training Accuracy": train_metrics_full["Training Accuracy"],
    "Training Micro F1": train_metrics_full["Training Micro F1"],
    "Training Macro F1": train_metrics_full["Training Macro F1"],
    "Training Precision@1": train_metrics_full["Training Precision@1"],
    "Training Precision@3": train_metrics_full["Training Precision@3"],
    "Training Precision@5": train_metrics_full["Training Precision@5"],

    "Validation Loss": val_metrics_full["Validation Loss"],
    "Validation Accuracy": val_metrics_full["Validation Accuracy"],
    "Validation Micro F1": val_metrics_full["Validation Micro F1"],
    "Validation Macro F1": val_metrics_full["Validation Macro F1"],
    "Validation Precision@1": val_metrics_full["Validation Precision@1"],
    "Validation Precision@3": val_metrics_full["Validation Precision@3"],
    "Validation Precision@5": val_metrics_full["Validation Precision@5"],

    "Testing Loss": test_metrics_full["Testing Loss"],
    "Testing Accuracy": test_metrics_full["Testing Accuracy"],
    "Testing Micro F1": test_metrics_full["Testing Micro F1"],
    "Testing Macro F1": test_metrics_full["Testing Macro F1"],
    "Testing Precision@1": test_metrics_full["Testing Precision@1"],
    "Testing Precision@3": test_metrics_full["Testing Precision@3"],
    "Testing Precision@5": test_metrics_full["Testing Precision@5"],
}

#### Results Table

In [17]:
results_df = pd.DataFrame({
    "Metric": [
        "Loss",
        "Accuracy",
        "Micro F1",
        "Macro F1",
        "Precision@1",
        "Precision@3",
        "Precision@5"
    ],
    "Training": [
        train_metrics_full["Training Loss"],
        train_metrics_full["Training Accuracy"],
        train_metrics_full["Training Micro F1"],
        train_metrics_full["Training Macro F1"],
        train_metrics_full["Training Precision@1"],
        train_metrics_full["Training Precision@3"],
        train_metrics_full["Training Precision@5"],
    ],
    "Validation": [
        val_metrics_full["Validation Loss"],
        val_metrics_full["Validation Accuracy"],
        val_metrics_full["Validation Micro F1"],
        val_metrics_full["Validation Macro F1"],
        val_metrics_full["Validation Precision@1"],
        val_metrics_full["Validation Precision@3"],
        val_metrics_full["Validation Precision@5"],
    ],
    "Testing": [
        test_metrics_full["Testing Loss"],
        test_metrics_full["Testing Accuracy"],
        test_metrics_full["Testing Micro F1"],
        test_metrics_full["Testing Macro F1"],
        test_metrics_full["Testing Precision@1"],
        test_metrics_full["Testing Precision@3"],
        test_metrics_full["Testing Precision@5"],
    ]
})

display(results_df)

,Metric,Training,Validation,Testing
0,Loss,1.3057,1.0913,1.0710
1,Accuracy,0.7374,0.7656,0.7647
2,Micro F1,0.7374,0.7656,0.7647
3,Macro F1,0.5291,0.5170,0.5129
4,Precision@1,0.7380,0.7657,0.7645
5,Precision@3,0.8352,0.8558,0.8576
6,Precision@5,0.8592,0.8849,0.8868


#### Save Results

In [18]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

json_path = os.path.join(OUTPUT_DIR, "final_metrics.json")
csv_path = os.path.join(OUTPUT_DIR, "final_metrics.csv")

with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

results_df.to_csv(csv_path, index=False)

print("Saved JSON:", json_path)
print("Saved CSV :", csv_path)

Saved JSON: ./clinical_bert_results\final_metrics.json
Saved CSV : ./clinical_bert_results\final_metrics.csv


#### Perplexity

In [19]:
perplexity_results = {
    "Train Perplexity": round(compute_perplexity(final_results["Training Loss"]), 4),
    "Validation Perplexity": round(compute_perplexity(final_results["Validation Loss"]), 4),
    "Test Perplexity": round(compute_perplexity(final_results["Testing Loss"]), 4),
}

perplexity_df = pd.DataFrame([perplexity_results])
display(perplexity_df)

,Train Perplexity,Validation Perplexity,Test Perplexity
0,3.6903,2.9781,2.9183


#### Final Summary

In [20]:
summary_df = pd.DataFrame({
    "Split": ["Training", "Validation", "Testing"],
    "Loss": [
        final_results["Training Loss"],
        final_results["Validation Loss"],
        final_results["Testing Loss"]
    ],
    "Accuracy": [
        final_results["Training Accuracy"],
        final_results["Validation Accuracy"],
        final_results["Testing Accuracy"]
    ],
    "Micro F1": [
        final_results["Training Micro F1"],
        final_results["Validation Micro F1"],
        final_results["Testing Micro F1"]
    ],
    "Macro F1": [
        final_results["Training Macro F1"],
        final_results["Validation Macro F1"],
        final_results["Testing Macro F1"]
    ],
    "Precision@1": [
        final_results["Training Precision@1"],
        final_results["Validation Precision@1"],
        final_results["Testing Precision@1"]
    ],
    "Precision@3": [
        final_results["Training Precision@3"],
        final_results["Validation Precision@3"],
        final_results["Testing Precision@3"]
    ],
    "Precision@5": [
        final_results["Training Precision@5"],
        final_results["Validation Precision@5"],
        final_results["Testing Precision@5"]
    ]
})

display(summary_df)

,Split,Loss,Accuracy,Micro F1,Macro F1,Precision@1,Precision@3,Precision@5
0,Training,1.3057,0.7374,0.7374,0.5291,0.7380,0.8352,0.8592
1,Validation,1.0913,0.7656,0.7656,0.5170,0.7657,0.8558,0.8849
2,Testing,1.0710,0.7647,0.7647,0.5129,0.7645,0.8576,0.8868
